# Pull neuroimaging data from workstation to local machine
This notebook demonstrates how to pull neuroimaging (rfMRI and dMRI) data of preselected subjects (`00_initial_cohort_selection.ipynb`; `01_cohort_matching.ipynb`) from a remote workstation to your local machine using bash commands, rsync and sshpass (for automatic password authentication). The notebook includes instructions for setting up the necessary tools and provides example commands for pulling the data. It also emphasizes the importance of securely handling credentials and ensuring that the data transfer is performed in compliance with relevant data protection regulations. This step is crucial for subsequent analyses that require access to the neuroimaging data, such as those performed in `04_merge_aggregate_rfMRI.ipynb` and `05_aggregate_dMRI.ipynb`. Due to the sensitive nature of the data and the need for secure handling, this notebook does not include actual commands with specific paths or credentials, but rather provides a template and guidelines for performing the data transfer from a remote workstation that has the necessary UK Biobank data access and permissions.

## Create necessary directories 
This section shows how to create directories (and subdirectories) based on your list of files to be pulled (containing subject IDs and desired instances [e.g., 'i2' for first imaging visit]). This list is generated in the previous notebook (01_cohort_matching.ipynb) for the control cohort and saved as a txt file. For the depression cohort, you can use the list generated in 00_initial_cohort_selection.ipynb.

File must contain entries in the following format:

subjecteID/i2

subjecteID/i2

...

In [ ]:
%%bash
BASE_DIR=".../data/UKB/control_notask_STRCO_RSSCHA_RSTIA" # Base directory on local machine (change according to which cohort you want to make)
FILE_LIST=".../data/UKB/cohorts/control_subject_ids.txt" # Path to file containing list of directories (and subdirectories) to make (change according to which cohort you want to work with)

while IFS= read -r entry || [ -n "$entry" ]; do
    full_dir="${BASE_DIR}/${entry}"
    
    if [ ! -d "$full_dir" ]; then
        mkdir -p "$full_dir"
        echo "Created directory: $full_dir"
    else 
        echo "Directory already exists: $full_dir"
    fi
done < "$FILE_LIST"

## Defining variables common to all cells below
Variables defined in a %%bash cell stay inside that specific subshell and dissapear when the cell execution is finished. Therefore, we define them in a python cell which makes them available in the 'global' scope of the notebook. They can then be accessed in any %%bash cell using the syntax "$VARIABLE_NAME".

In [ ]:
import getpass
import os

# Configuration

config = {
    "REMOTE_SOURCE": ".../fMRI_rest/ts/Tian", # change according to which dataset you are pulling
    "LOCAL_DESTINATION": ".../data/UKB/control_notask_STRCO_RSSCHA_RSTIA", # change according to which cohort you want to make
    "FILE_LIST_STR": ".../data/UKB/cohorts/control_subject_ids.txt", # change according to which cohort you want to work with
    "FILE_LIST_RSSCHA": ".../data/UKB/cohorts/control_subject_ids_with_schaefer_suffix.txt", # change according to which cohort you want to work with (you might need to first execute a cell below to create this file by adding the Schaefer suffix to the subject IDs)
    "FILE_LIST_RSTIA": ".../data/UKB/cohorts/control_subject_ids_with_tian_suffix.txt", # change according to which cohort you want to work with (you might need to first execute a cell below to create this file by adding the Tian suffix to the subject IDs)
    "LOG_FILE": "missing_files_log.txt",
    "TEMP_LOG": "temp_missing_files.txt"
}

# Inject into the environment so Bash cells can access them as $VARIABLES
for key, value in config.items():
    os.environ[key] = value

print("✅ Configuration variables loaded and exported to environment.")

# This creates a secure, masked input field in the notebook UI
password = getpass.getpass("Enter SSH password: ")

# We push this to the environment so the next Bash cell can see it
os.environ['SSHPASS'] = password
print("✅ Password stored in environment for this session.")

## Check for missing files or directories before pull
This section checks for any missing files or directories in the remote workstation directories before initiating the pull process. It lists any subjects that do not have the expected rfMRI or dMRI files.

In [ ]:
%%bash
echo "Starting check..."

# Use -e to tell sshpass to use the SSHPASS environment variable
# Use -o StrictHostKeyChecking=no to prevent the "Unknown Host" interactive prompt
sshpass -e rsync -av \
    -e "ssh -o StrictHostKeyChecking=no" \
    --files-from="$FILE_LIST_STR" \
    --exclude '*/' \
    "${REMOTE_SOURCE}" "${LOCAL_DESTINATION}" > "$TEMP_LOG" 2>&1

# Parse results
grep -E "No such file or directory|failed" "$TEMP_LOG" > "$LOG_FILE" || true
rm "$TEMP_LOG"

echo "Transfer complete. Check $LOG_FILE for any errors."

## Adding other suffixes to subject IDs
For the extraction of resting-state fMRI cortical and subcortical data, we need to add the specific filenames to the subject IDs and instance code (i2). We add these filenames because we do not want to extract entire subject folders for the resting-state data as this data is large (entire timeseries, not already computed structural matrices). The suffixes are as follows:
- '.../fMRI.Tian_Subcortex_S4_3T.csv.gz' for resting-state fMRI subcortical data
- '.../fMRI.Schaefer7n1000p.csv.gz' for resting-state fMRI cortical data

In [ ]:
%%bash
# append literal suffix to every line, write to new file
sed 's|$|/fMRI.Schaefer7n1000p.csv.gz|' "$FILE_LIST_STR" > "${FILE_LIST_STR%.txt}_with_schaefer_suffix.txt"
sed 's|$|/fMRI.Tian_Subcortex_S4_3T.csv.gz|' "$FILE_LIST_STR" > "${FILE_LIST_STR%.txt}_with_tian_suffix.txt"

## Pull files or entire directories
This section demonstrates how to pull specific neuroimaging files (rfMRI and dMRI) or entire directories (e.g., entire subject folders) from the remote workstation to your local machine using rsync and sshpass for automatic password authentication.

In [ ]:
%%bash
# Check if variables are present; if not, exit to prevent errors
if [ -z "$FILE_LIST_RSTIA" ]; then
    echo "Error: Configuration variables not found. Please run the Python config cell first."
    exit 1
fi

echo "Starting batch sync from: $FILE_LIST_RSTIA"

# The loop reads each line from the file list
while read -r dir || [ -n "$dir" ]; do 
    # Skip empty lines or comments
    [[ -z "$dir" || "$dir" =~ ^# ]] && continue

    echo "------------------------------------------"
    echo "Syncing: ${dir} ..."
    
    # Use -o StrictHostKeyChecking=no to prevent hanging on new connections
    # We use ${REMOTE_SOURCE}${dir} to build the full remote path
    sshpass -e rsync -av -e "ssh -o StrictHostKeyChecking=no" \
        "${REMOTE_SOURCE}/${dir}" \
        "${LOCAL_DESTINATION}/${dir}"
        
done < "$FILE_LIST_RSTIA"

echo "------------------------------------------"
echo "Data pull complete."

# Unset within the subshell session
unset SSHPASS

In [ ]:
# Delete the SSH password from the notebook's memory
del os.environ['SSHPASS']